In [1]:
cd ..

/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis


In [2]:
import numpy as np 
import pandas as pd 
import requests
import json
import pytz
from datetime import datetime
from multithread_processing.base_job import BaseJob
from utils.logger_utils import get_logger
from databases.mongodb_main import MongoDBMain



In [3]:
##Filter hot_burn
# with open('data/special_wallets/burn_wallets.json', 'r') as file:
#     burn_data = json.load(file)

# with open('data/special_wallets/hot_wallets.json', 'r') as file:
#     hot_data = json.load(file)

# burn_addresses = []    
# hot_addresses = []    

# for _, burn_value in burn_data.items():
#     burn_addresses.extend(burn_value)
# for _, hot_name in hot_data.items():
#     value_lst = list(hot_name.values())
#     addresses_lst = value_lst[1]
#     for _, hot_address in addresses_lst.items():
#         hot_addresses.extend(hot_address)
# burn_addresses = set(burn_addresses)
# burn_addresses.update(hot_addresses)
# hot_burn_addresses = set(burn_addresses)

In [4]:
class GetTransactionInfo(BaseJob):
    def __init__(self, chain, collections, historical_token_price, wallets_list: list, filename: str, start_date, end_date,
                 max_workers, batch_size):
        
        mongo = MongoDBMain(chain=chain)
        self.transac_collection = mongo._db[collections]        
        self.transaction_info = list()
        self.transfer_info = list()
        self.filename = filename
        self.start_date = start_date
        self.end_date = end_date
        self.historical_token_price = historical_token_price
        super().__init__(work_iterable=wallets_list,
                         max_workers=max_workers,
                         batch_size=batch_size)

    def _execute_batch(self, works):
        df_list = list()
        cursor =self.transac_collection.find({"block_timestamp" : {"$gte": self.start_date, "$lte": self.end_date},'from_address': {'$in': works}},{'from_address':1, 'to_address':1, 'value':1, 'block_timestamp':1,'transaction_type':1})
        for doc in cursor:
            single_transaction_info = dict(doc) 
            if single_transaction_info['value'] == '0':
                continue
            else:
                round_timestamp = (single_transaction_info['block_timestamp']  - single_transaction_info['block_timestamp']  % 86400) *1000
                # round_timestamp = (single_transaction_info['block_timestamp'] // 3600) * 3600000
                single_transaction_info['valueInUSD'] = int(single_transaction_info['value']) * self.historical_token_price['ethereum'][round_timestamp]
                df_list.append(single_transaction_info)    

        df_concat = pd.DataFrame(df_list)
        self.transaction_info.append(df_concat)
        
    def _end(self):
        super()._end()
        df = pd.concat(self.transaction_info, ignore_index=True)
        #return df
        df.to_csv(self.filename, index=False)


In [5]:
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=90, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        prices = data.get('prices', [])
        formatted_prices = format_prices(prices)
        prices_dict = {token_id: formatted_prices}
        return prices_dict
    else:
        return None

In [6]:
historical_token_price = get_historical_prices('ethereum')
historical_token_round_price = dict()
historical_token_round_price['ethereum'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['ethereum'].items()}


In [9]:
logger = get_logger('User infromation exporter')
input_df = pd.read_json('data/reputation_score/aave_reputation.json')
addresses = input_df.wallet.unique().tolist()[0:5]

In [7]:

job = GetTransactionInfo(chain='ethereum', 
                            collections='transactions', 
                            historical_token_price=historical_token_round_price,
                            wallets_list= addresses, 
                            filename='data/transaction_info/test.csv', 
                            start_date=1701388800, 
                            end_date= 1708992000, 
                            max_workers=4, 
                            batch_size=10)
job.run()


[02-28-2024 15:53:03 +07] [INFO] [BatchWorkExecutor] - Started work. Items to process: 100.


[02-28-2024 15:53:10 +07] [INFO] [BatchWorkExecutor] - 15 items processed. Progress is 15%.
[02-28-2024 15:53:16 +07] [INFO] [BatchWorkExecutor] - 30 items processed. Progress is 30%.
[02-28-2024 15:53:17 +07] [INFO] [BatchWorkExecutor] - 45 items processed. Progress is 45%.
[02-28-2024 15:53:18 +07] [INFO] [BatchWorkExecutor] - 60 items processed. Progress is 60%.
[02-28-2024 15:53:19 +07] [INFO] [BatchWorkExecutor] - 75 items processed. Progress is 75%.
[02-28-2024 15:53:22 +07] [INFO] [BatchWorkExecutor] - 90 items processed. Progress is 90%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - 100 items processed. Progress is 100%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - Finished work. Total items processed: 100. Took 0:00:20.554740.


## test

In [4]:
import pandas as pd
df = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transaction_info/test_receiving_groupby.csv")
df

,receiving_wallet,wallet,block_timestamp_tuple,valueInUSD_sum,valueInUSD_median,NumberOfTokensBorrow,NumberOfTokensDeposit,NumberOfTokensHold,totalBorrowInUSD,totalDepositInUSD,totalHoldInUSD,score
0,0x47773580f5de4b2052f94b8b142d71f4efa29f3d,0x3744da57184575064838bbc87a0fc791f5e39ea2,"(1701808127,)",6.062422e+22,6.062422e+22,0,0,1,0.000000e+00,0.000000e+00,2.108893e+07,40468.188820
1,0x545335cafe74b4cea332c6e2c3a557cd93fc8063,0x3744da57184575064838bbc87a0fc791f5e39ea2,"(1706209355,)",6.410310e+15,6.410310e+15,0,0,1,0.000000e+00,0.000000e+00,2.108893e+07,40468.188820
2,0x60ac32d416507c844b43e29b2e12dd23f5277420,0x3744da57184575064838bbc87a0fc791f5e39ea2,"(1705539611,)",4.897518e+22,4.897518e+22,0,0,1,0.000000e+00,0.000000e+00,2.108893e+07,40468.188820
3,0x063ed6f59bd44d8bc99c3b170a3d52b49dcbcfff,0x66a00690bad92cc6bb02d52ae7ed3c2fb671a29a,"(1707782411,)",5.406290e+23,5.406290e+23,0,0,1,0.000000e+00,0.000000e+00,1.929168e+06,21872.378405
4,0x21a31ee1afc51d94c2efccaa2092ad1028285549,0x64b6ebe0a55244f09dfb1e46fe59b74ab94f8be1,"(1702355627,)",1.166985e+24,1.166985e+24,3,2,0,8.499525e+06,6.188003e+07,0.000000e+00,26237.059098
...,...,...,...,...,...,...,...,...,...,...,...,...
4217,0xff9962dcddc7f839d60090740a84467a94352ac4,0x5770815b0c2a09a43c9e5aecb7e2f3886075b605,"(1706803487, 1705829531, 1706519615, 170584463...",2.377352e+21,2.583499e+20,0,0,1,0.000000e+00,0.000000e+00,6.069155e+06,25890.507175
4218,0xff9f7ec6e02bccd4d725b809cbd2410de22c10f1,0x5770815b0c2a09a43c9e5aecb7e2f3886075b605,"(1701430571,)",8.768773e+20,8.768773e+20,0,0,1,0.000000e+00,0.000000e+00,6.069155e+06,25890.507175
4219,0xffaca3564399c2603fcd76295da68e3c654a6fac,0x5770815b0c2a09a43c9e5aecb7e2f3886075b605,"(1706042207,)",1.351217e+22,1.351217e+22,0,0,1,0.000000e+00,0.000000e+00,6.069155e+06,25890.507175
4220,0xffc3884e5028fb279dbefe83dd18329c8fd0419d,0x5770815b0c2a09a43c9e5aecb7e2f3886075b605,"(1703700143,)",1.332791e+21,1.332791e+21,0,0,1,0.000000e+00,0.000000e+00,6.069155e+06,25890.507175


##  Postgres test

In [6]:
blockchain_etl = MongoDBMain(chain='ethereum')

block_timestamp_mappings = blockchain_etl.get_block_number_to_timestamp(start_block=18687851,
                                                                                                      end_block=19129889)

In [22]:
from databases.postgresql import PostgresDB
postgres = PostgresDB()
transfers_data = postgres.query_addresses(addresses=addresses,from_block=18687851,
                                                        to_block=19129889,
                                                        chain_id='0x1')
output_data = []
fields_to_select =['contract_address', 'block_number', 'from_address', 'to_address', 'value']
for row in transfers_data:
    new_row = row[:1] + (block_timestamp_mappings[row[1]],) +row[2:]

    output_data.append(dict(zip(fields_to_select, new_row)))

### Mongo Entity

In [2]:
from databases.mongodb_entity import MongoDBEntity


In [8]:
input_df = pd.read_json('data/reputation_score_27_2/aave_reputation.json')
input_address = input_df.wallet.unique().tolist()[0:5]
addresses = []
for address in input_address:
    addresses.append('0x1_'+address)


In [9]:
addresses

['0x1_0x3744da57184575064838bbc87a0fc791f5e39ea2',
 '0x1_0x5a801a9418d036fd453078c3adcb761fdc5ae695',
 '0x1_0x3555ef98046fac600c0b6529e7018ebeca176398',
 '0x1_0x61b0e6b68184eb0316b1285c4e76a15bfc7cd857',
 '0x1_0x80845058350b8c3df5c3015d8a717d64b3bf9267']

In [14]:
mongo_klg = MongoDBEntity()
mongo_klg_wallet=mongo_klg.wallets
wallets = mongo_klg_wallet.find({'_id': {'$in': addresses}})
wallet_lst = []
for wallet in wallets:
    wallet_lst.append(wallet)
wallet_df = pd.DataFrame(wallet_lst)
wallet_df.columns

Index(['_id', 'address', 'chainId', 'newElite', 'borrowChangeLogs', 'lendings',
       'depositInUSD', 'borrowTokens', 'lastUpdatedAt', 'depositChangeLogs',
       'tokenChangeLogs', 'balanceChangeLogs', 'depositTokens',
       'borrowTokenChangeLogs', 'borrowInUSD', 'tokens', 'balanceInUSD',
       'depositTokenChangeLogs', 'elite', 'dailyNumberOfTransactions',
       'dailyTransactionAmounts', 'numberCallsTopContracts', 'liquidationLogs',
       'numberOfLiquidation', 'totalValueOfLiquidation',
       'dailyAllTransactions', 'flagged', 'newExtend', 'createdAt',
       'frequencyOfDappTransactions', 'numberOfInteractedDapps',
       'typesOfInteractedDapps', 'numberOfReputableDapps', 'tokensInteraction',
       'tags', 'type', 'newTarget', 'lastTokenInteractedAt',
       'lastSyncedDataAt', 'lastSyncedFullDataAt', 'nfts', 'rewards', 'vaults',
       'dexes', 'selective', 'chainName', 'isNew', 'writingLock'],
      dtype='object')

In [22]:
import requests
import time
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=92, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    print(response.status_code)
    if response.status_code == 200:
        time.sleep(3)
    elif response.status_code == 429:
        time.sleep(60)
        response = requests.get(url)
    else:
        return None
    
    data = response.json()
    prices = data.get('prices', [])
    formatted_prices = format_prices(prices)
    prices_dict = {token_id: formatted_prices}
    return prices_dict

In [78]:
aave_from = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transfer_info/from_transfer.csv")
aave_to = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transfer_info/to_transfer.csv")


In [ ]:
import json

In [48]:
token_from = aave_from.contract_address.unique().tolist()
token_to = aave_to.contract_address.unique().tolist()
token_from.extend(token_to)
tokens = list(set(token_from))
with open('/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/tokens/aave_tokens.json', "w") as json_file:
    json.dump(tokens, json_file, indent=4)

In [49]:
with open('/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/tokens/aave_tokens.json', "r") as json_file:
    data = json.load(json_file)

In [28]:
historical_token_price = get_historical_prices('airnft-token')
print(historical_token_price)
historical_token_round_price = dict()

historical_token_round_price['airnft-token'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['airnft-token'].items()}

429
{'airnft-token': {1701475200000: 0.0012436522456428483, 1701561600000: 0.0012494789766214203, 1701648000000: 0.001249378142686813, 1701734400000: 0.0012575906762466345, 1701820800000: 0.0012410994152013282, 1701907200000: 0.0011410766413160096, 1701993600000: 0.0011718880449486596, 1702080000000: 0.0011990750129799103, 1702166400000: 0.0011982851539001946, 1702252800000: 0.001214548899103537, 1702339200000: 0.0011068480136135431, 1702425600000: 0.0010085863035178386, 1702512000000: 0.0009986413628549464, 1702598400000: 0.0010281176711467292, 1702684800000: 0.0009668426111913278, 1702771200000: 0.0009639195032261808, 1702857600000: 0.0009527960207278551, 1702944000000: 0.0009342008602369286, 1703030400000: 0.0009710455200180209, 1703116800000: 0.0010385634372220562, 1703203200000: 0.001029199910777607, 1703289600000: 0.0010123299825562078, 1703376000000: 0.0010272573947850657, 1703462400000: 0.0009800739430138835, 1703548800000: 0.0009772396451878458, 1703635200000: 0.00109218520260